<a href="https://colab.research.google.com/github/LovishJaswal/GenAI-workspace/blob/python_tasks/Tooling_Task_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ddgs
!pip install -q langchain-community pdfplumber langchain-text-splitters langchain-huggingface langchain-chroma openai
!pip install -q validators
!pip install -q bs4
!pip install -q sentence-transformers

In [ ]:
import gradio as gr
from openai import OpenAI
from ddgs import DDGS
from google.colab import userdata
import json
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from datetime import datetime
from zoneinfo import ZoneInfo
import time
import requests
import validators
from bs4 import BeautifulSoup

/tmp/ipykernel_18112/2141660945.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PDFPlumberLoader


In [ ]:
api_key = userdata.get('api_key')

In [ ]:
client = OpenAI(
    base_url = "https://openrouter.ai/api/v1",
    api_key = api_key,
    )

In [ ]:
def search_web(query):
  results = DDGS().text(query)
  return results


In [ ]:
USER_TIMEZONE = "Asia/Kolkata"

def get_current_datetime():
    now = datetime.now(ZoneInfo(USER_TIMEZONE))

    return {
        "datetime": now.isoformat(timespec="seconds"),
        "date": now.strftime("%Y-%m-%d"),
        "time": now.strftime("%H:%M:%S"),
        "day": now.strftime("%A"),
        "timezone": USER_TIMEZONE
    }

In [ ]:
def web_scraping(query_or_url):
    if validators.url(query_or_url):
        try:
            response = requests.get(query_or_url, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')
            return soup.get_text(separator=' ', strip=True)[:10000]
        except Exception as e:
            return f"Error scraping URL: {str(e)}"
    else:
        return "Error: Please provide a valid URL, not a search query."

In [ ]:
def retrieve_data(query):
    global vector_db

    if vector_db is None:
        return {"error": "No PDF uploaded."}

    docs = vector_db.search(
        query=query,
        search_type="mmr",
        k=5
    )
    messages[0]["content"] += """

    A PDF has been uploaded and indexed successfully.

    From now on:
    - If the user refers to "this PDF", "the document", "the uploaded file",
      "according to the PDF", or asks about information that could be in the uploaded
      document, ALWAYS call the `retrieve_data` tool before answering.
    - Do not answer from your own knowledge for questions about the uploaded document.
    """

    return [
        {
            "content": doc.page_content,
            "metadata": doc.metadata
        }
        for doc in docs
    ]

In [ ]:
system_prompt = {
    "role": "system",
    "content": """
You are a helpful AI assistant with access to external tools.

Your responsibility is to determine whether a user's request requires one or more tools before answering.

## Tool Usage Guidelines

### 1. search_web
Use `search_web` whenever the user asks about:
- latest or breaking news
- current events
- weather
- live sports scores
- stock prices
- recent political, business, or technology developments
- information that may have changed since your training
- factual information that requires up-to-date internet access
- finding websites, articles, blogs, documentation, or online resources

### 2. scrape_webpage
Use `scrape_webpage` whenever the user provides:
- a webpage URL
- a blog link
- an article link
- documentation link
- any webpage whose contents need to be read

Use this tool to:
- summarize webpage contents
- extract specific information
- answer questions about a webpage
- analyze webpage text
- compare information across webpages
- extract tables, FAQs, lists, or structured content

If multiple webpages are provided, scrape each relevant page before answering.

### 3. get_current_datetime
Use `get_current_datetime` whenever the user asks about:
- the current date
- today's date
- the current time
- today's day
- local date or time

### 4. retrieve_data
Use `retrieve_data` whenever the user asks about information contained in uploaded documents or PDFs, including:
- document summaries
- chapter explanations
- answering questions from the document
- locating specific facts
- retrieving relevant passages
- explaining sections or topics

### 5. Multiple Tools
If the user's request requires multiple tools, call all relevant tools before answering.

Examples:
- "Summarize this article and tell me today's date." → scrape_webpage + get_current_datetime
- "Compare the latest AI news with this blog post." → search_web + scrape_webpage
- "Answer this question using my PDF and the latest news." → retrieve_data + search_web

### 6. Clarification
If the user's request is ambiguous or lacks enough information (such as missing URLs, document references, or unclear intent), ask a clarifying question before using any tool.

### 7. Response Generation
After receiving tool outputs:
- Combine all relevant information into one coherent answer.
- Do not mention internal reasoning or tool selection unless the user explicitly asks.
- Clearly distinguish between retrieved information and general knowledge when appropriate.

### 8. Missing Information
If:
- the retrieval tool finds no relevant document content, state that the information is not present in the uploaded document.
- the webpage cannot be scraped, inform the user politely instead of fabricating information.
- web search finds insufficient information, explain the limitation honestly.

### 9. General Behavior
- Always prioritize correctness.
- Follow the user's intent.
- Use tools only when necessary.
- Never fabricate tool outputs.
- Maintain a professional, helpful, and concise tone.
"""
}

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": (
                "Search the internet for up-to-date information that is not available "
                "from the model's internal knowledge. Use this tool for latest news, "
                "weather, live sports scores, stock prices, recent events, current "
                "political developments, recent product information, or any question "
                "that requires information from the web. Do NOT use this tool to get "
                "the current date or current time; use get_current_datetime instead."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A concise search query describing the information to retrieve."
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_datetime",
            "description": (
                "Return the user's current local date, time, day of the week, and "
                "current datetime. Use this whenever the user asks for today's date, "
                "the current time, the current day, or any question requiring the "
                "current local date and time. Do NOT use search_web for these requests."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "retrieve_data",
            "description": (
                "Search the uploaded PDF knowledge base and retrieve the most relevant "
                "information for the user's question. Use this tool whenever the user "
                "asks about the contents of an uploaded document, PDF, report, paper, "
                "manual, notes, or any information that may exist in the uploaded "
                "knowledge base. Do NOT use this tool for general knowledge or current "
                "events. If no document has been uploaded, the tool will indicate that."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": (
                            "The user's question or a semantic search query used to "
                            "retrieve relevant document chunks."
                        )
                    }
                },
                "required": ["query"]
            }
        }
    },
      {
        "type": "function",
        "function": {
            "name": "web_scraping",
            "description": "Get response according to the user's query by searching from web",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A search query string",
                    },
                },
                "required": ['query'],
            },
        },
    }
]

In [ ]:
encoder = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2"
    )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
def upload_pdf(pdf, progress=gr.Progress()):
    global vector_db

    progress(0.05, desc="📄 Upload received...")
    time.sleep(0.3)

    progress(0.15, desc="🔍 Extracting text from PDF...")
    loader = PDFPlumberLoader(pdf.name)
    documents = loader.load()

    progress(0.35, desc="✂️ Splitting document into semantic chunks...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )
    chunks = text_splitter.split_documents(documents)

    progress(0.60, desc="🧠 Computing vector embeddings...")
    time.sleep(0.2)

    progress(0.75, desc="📚 Please wait! Loading in progress...This could take upto 1 min")
    vector_db = Chroma(
        embedding_function=encoder,
        persist_directory="db",
        collection_name="vectorDB"
    )
    vector_db.add_documents(chunks)

    progress(0.95, desc="⚡ Optimizing retrieval index...")
    time.sleep(0.2)

    progress(1.0, desc="✅ Knowledge Base Ready!")

    return f"""

**📄 Document:** `{pdf.name.split('/')[-1]}`

**📚 Chunks Indexed:** `{len(chunks)}`

Your document has been successfully processed and indexed.

You can now:
- 💬 Ask questions about the document
- 📝 Request summaries
- 🔍 Find specific topics or concepts
- 📖 Explain sections in detail
- 🎯 Extract important information

**Happy chatting! 🚀**
"""

In [ ]:
messages = [system_prompt]
def Chatbot(prompt, history):
  messages.append({"role": "user","content": prompt})

  response = client.chat.completions.create(
      model = "google/gemini-2.5-flash",
      messages=messages,
      tools=tools,
      max_tokens = 7000
  )
  if (response.choices[0].message.tool_calls):
    print(response.choices[0].message.tool_calls[0].function.arguments)

    messages.append(response.choices[0].message)

    for call in response.choices[0].message.tool_calls:
        print("The call is:", call.function.name)

        if call.function.name == "search_web":
            answer = search_web(
                json.loads(call.function.arguments)["query"]
            )

        elif call.function.name == "get_current_datetime":
            answer = get_current_datetime()

        elif call.function.name == "retrieve_data":
            answer = retrieve_data(
                json.loads(call.function.arguments)["query"]
            )
        elif call.function.name == "web_scraping":
            args = json.loads(call.function.arguments)
            answer = web_scraping(args["query"])

        messages.append(
            {
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(answer)
            }
        )

    final_response = client.chat.completions.create(
        model="google/gemini-2.5-flash",
        messages=messages,
        max_tokens = 6000
    )

    messages.append({"role": "assistant", "content": final_response.choices[0].message.content})
    return final_response.choices[0].message.content
  else:
    messages.append({"role": "assistant", "content": response.choices[0].message.content})
    return response.choices[0].message.content

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown("#AI Assistant with RAG")

    pdf = gr.File(
        label="Upload PDF",
        file_types=[".pdf"]
    )

    status = gr.Markdown("**Status:** No document uploaded.")
    pdf.change(
        fn=upload_pdf,
        inputs=pdf,
        outputs=status
    )

    gr.ChatInterface(
        fn=Chatbot,
        title="Chat"
    )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a32daee7097f88c71b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


{}
The call is: get_current_datetime


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a32daee7097f88c71b.gradio.live
